# 13 — RQ2 model-family decomposition

**Objective.** Compare inter- versus intra-family attribution disagreement, prediction-equivalent pairs, crossed explanation uncertainty, and construct-robust evidence states.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

The decomposition is design-specific and descriptive; finite outcomes/model families are not random samples from all possible constructs or algorithms.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("13", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import pandas as pd
import yaml
from cruxvc.inference import (
    attribution_vectors,
    bootstrap_interval,
    bootstrap_rq2_delta_family,
    construct_driver_taxonomy,
    fit_crossed_attribution_models,
    model_pair_distances,
    rq2_delta_family,
    three_way_interpretation,
)
from cruxvc.io import read_table, write_table
from cruxvc.manifest import append_test_access_log

append_test_access_log(
    P,
    stage_id="13",
    purpose="locked RQ2/decomposition inference",
    resources=[
        P.attributions / "final_attributions_long.parquet",
        P.predictions / "final_predictions.parquet",
    ],
)
long = read_table(P.attributions / "final_attributions_long.parquet")
predictions = read_table(P.predictions / "final_predictions.parquet")
# RQ2 uses the label-specific near-optimal sets only. Matched-reference deployment
# rows duplicate a subset of these fitted configurations and would otherwise
# distort pair weighting.
deployment_attr = long[long["analysis_role"].eq("label_specific_near_optimal_deployment")].copy()
deployment_pred = predictions[
    predictions["analysis_role"].eq("label_specific_near_optimal_deployment")
].copy()
wide = attribution_vectors(deployment_attr)
# The crossed decomposition is the controlled matched-reference design, including
# frozen bootstrap refits and the deployment fit, not the duplicated tuning set.
decomposition_attr = long[
    long["analysis_role"].isin(["matched_reference_bootstrap", "matched_reference_deployment"])
].copy()
statistical = yaml.safe_load((P.config / "statistical_analysis.yaml").read_text(encoding="utf-8"))

In [ ]:
pairs = model_pair_distances(
    wide,
    prediction_table=deployment_pred,
    probability_tolerance=float(statistical["inference"]["prediction_equivalence_probability_tolerance"]),
)
bootstrap_repetitions = int(statistical["inference"]["bootstrap_repetitions"])
bootstrap_seed = int(statistical["inference"]["bootstrap_seed"]) + 130
summary_parts = []
draw_parts = []
for population_index, (population, population_pairs) in enumerate([
    ("all_near_optimal_pairs", pairs),
    ("prediction_equivalent_pairs", pairs[pairs["prediction_equivalent"].fillna(False)]),
]):
    point = rq2_delta_family(population_pairs).assign(population=population)
    draws = bootstrap_rq2_delta_family(
        population_pairs,
        repetitions=bootstrap_repetitions,
        seed=bootstrap_seed + population_index,
    ).assign(population=population)
    intervals = []
    for outcome, part in draws.groupby("outcome"):
        interval = bootstrap_interval(part["delta_family"])
        estimate = float(point.loc[point["outcome"].eq(outcome), "delta_family"].iloc[0])
        intervals.append({
            "outcome": outcome,
            "ci_lower": interval[0],
            "ci_upper": interval[1],
            "decision": three_way_interpretation(
                estimate,
                interval,
                meaningful_effect=float(statistical["smallest_meaningful_effects"]["rq2_delta_family"]),
                equivalence_half_width=float(statistical["power"]["equivalence_half_width"]),
            ),
            "bootstrap_repetitions": int(part["delta_family"].notna().sum()),
        })
    point = point.merge(pd.DataFrame(intervals), on="outcome", how="left", validate="one_to_one")
    summary_parts.append(point)
    draw_parts.append(draws)
summary = pd.concat(summary_parts, ignore_index=True)
bootstrap_draws = pd.concat(draw_parts, ignore_index=True)
pair_path = write_table(pairs, P.inference / "rq2_model_pair_distances.parquet")
summary_path = write_table(summary, P.inference / "rq2_family_effects.csv")
draws_path = write_table(bootstrap_draws, P.inference / "rq2_bootstrap_draws.parquet")

In [ ]:
decomposition = fit_crossed_attribution_models(decomposition_attr)
decomposition_path = write_table(
    decomposition,
    P.inference / "crossed_attribution_decomposition.csv",
)
# This pre-control map must remain unresolved. Notebook 14 replaces it with the
# control-gated final taxonomy only after all mandatory controls are evaluated.
precontrol_taxonomy = construct_driver_taxonomy(deployment_attr, faithfulness=None)
taxonomy_path = write_table(
    precontrol_taxonomy,
    P.inference / "construct_robust_driver_taxonomy_precontrols.csv",
)

In [ ]:
CTX.recorder.complete([pair_path, summary_path, draws_path, decomposition_path, taxonomy_path])
print(summary.to_string(index=False))
print(precontrol_taxonomy.to_string(index=False))